# Differential Chromatic Refraction (DCR) for LSST Deep Drilling Fields

## Objective

This notebook estimates the **theoretical dipole length** induced by Differential Chromatic Refraction (DCR)
for each LSST Deep Drilling Field (DDF), as a function of hour angle $H$.

### Physical model

In a plane-parallel atmosphere, the angular refraction shift at wavelength $\lambda$ and zenith angle $z$ is:
$$R(\lambda, z) \approx (n(\lambda) - 1)\, \tan z$$

For a source observed through band $b$, the relevant quantity is **not** the mean refraction, but the
**chromatic spread** across the band — i.e., the dispersion of $n(\lambda)$ weighted by the
photon-count response.

### Photon-count weighting

A CCD counts photons.  The number of photons detected per unit wavelength interval is:
$$\frac{dN_\gamma}{d\lambda} \propto T_b(\lambda)\cdot f_\lambda(\lambda)\cdot \frac{\lambda}{hc}$$
where $T_b(\lambda)$ is the total throughput, $f_\lambda$ the source flux per unit wavelength,
and $\lambda/(hc)$ converts photon energy to photon count.

With a **flat SED in $f_\nu$** ($f_\nu = \text{const}$), we have
$f_\lambda = f_\nu / \lambda^2$, so:
$$\frac{dN_\gamma}{d\lambda} \propto T_b(\lambda)\cdot \frac{f_\nu}{\lambda^2}\cdot \lambda
  = f_\nu\,\frac{T_b(\lambda)}{\lambda}$$

The integration weight therefore reduces to:
$$\boxed{w_b(\lambda) \propto \frac{T_b(\lambda)}{\lambda}}$$

### Band chromatic dispersion

The normalised weight is $\tilde{w}_b(\lambda) = w_b(\lambda)\,/\,\int w_b\,d\lambda$.

Band-averaged refractive index:
$$\langle n \rangle_b = \int \tilde{w}_b(\lambda)\, n(\lambda)\, d\lambda$$

Chromatic dispersion (weighted RMS of $n$ across the band):
$$\sigma_n(b) = \sqrt{\int \tilde{w}_b(\lambda)\, [n(\lambda) - \langle n \rangle_b]^2\, d\lambda}$$

Expected dipole length in arcseconds:
$$\boxed{l_{\rm dip}(b, z) = \sigma_n(b) \times \tan z \times \frac{180\times 3600}{\pi}}$$

### Layout
Figures show $l_{\rm dip}$ vs hour angle $H \in [-6\,{\rm h},\, +6\,{\rm h}]$ for each DDF,
in a **2 × 3 subplot grid**, one panel per field, all six LSST bands overlaid.

## Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import simpson

from speclite import filters
import ref_index

from astropy.coordinates import EarthLocation, SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")
print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

## Configuration

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
NB_TAG = "TOOLS_07_DDF-DCR"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

In [ ]:
# ── LSST Deep Drilling Fields (RA deg, Dec deg) ───────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# ── Conversion constants ───────────────────────────────────────────────────────
RAD_TO_ARCSEC = 180.0 / np.pi * 3600.0

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

## Refractive index — Ciddor formula

We use the `ref_index` package which implements the Ciddor (1996) formula for the
refractive index of moist air as a function of wavelength, temperature, pressure and
relative humidity.  Default conditions: $T = 20\,°\text{C}$, $P = 101325\,\text{Pa}$,
$\text{RH} = 20\%$.

In [ ]:
# Wavelength grid covering the full LSST range (0.3 – 1.1 µm)
LAM = np.linspace(0.3, 1.1, 2000)  # µm

In [ ]:
def n_ciddor(lam, t=20.0, p=101325.0, rh=20.0):
    """
    Refractive index of moist air via the Ciddor (1996) formula.

    Parameters
    ----------
    lam : array_like
        Wavelength in **microns**.
    t   : float
        Temperature in °C (default 20).
    p   : float
        Pressure in Pa (default 101 325).
    rh  : float
        Relative humidity in % (default 20).

    Returns
    -------
    n : ndarray
        Refractive index n(λ).
    """
    return ref_index.ciddor(wave=np.asarray(lam) * 1000.0, t=t, p=p, rh=rh)


# Pre-compute n(λ) on the global grid once
N_LAM = n_ciddor(LAM)

## LSST filter throughputs (speclite lsst2023)

In [ ]:
# Load all six LSST 2023 filter curves
BANDS = list("ugrizy")
_lsst_seq = filters.load_filters("lsst2023-*")
LSST_FILTERS = {f.name: f for f in _lsst_seq}  # keyed as 'lsst2023-u', etc.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
for band in BAND_ORDER:
    filter_name = f"lsst2023-{band}"
    throughput = LSST_FILTERS[filter_name]
    x = throughput.wavelength
    y = throughput.response
    ax.plot(x, y, color=BAND_COLORS[band], label=filter_name)
ax.legend(title="from speclite:", bbox_to_anchor=(1.1, 1.05))
ax.set_xlabel("$\lambda$ (nm)")
ax.set_title("LSSST throughput")
plt.show()

## Key function: `compute_sigma_n` — chromatic dispersion of the refractive index

### Photon-count weight (flat $f_\nu$ SED)

A CCD measures photon counts.  The detected photon rate per unit wavelength is:
$$\frac{dN_\gamma}{d\lambda} \propto T_b(\lambda)\cdot f_\lambda(\lambda)\cdot\frac{\lambda}{hc}$$

With a flat SED in $f_\nu$ ($f_\nu = \text{const}$):
$$f_\lambda = \frac{f_\nu}{\lambda^2}
\quad\Longrightarrow\quad
\frac{dN_\gamma}{d\lambda} \propto \frac{T_b(\lambda)}{\lambda^2}\cdot\lambda
= \frac{T_b(\lambda)}{\lambda}$$

The integration weight is therefore:
$$w_b(\lambda) = \frac{T_b(\lambda)}{\lambda}$$

### Band chromatic dispersion

Normalised weight: $\tilde{w}_b = w_b \,/\,\int w_b\,d\lambda$.

Band-averaged refractive index:
$$\langle n \rangle_b = \int \tilde{w}_b(\lambda)\, n(\lambda)\, d\lambda$$

Chromatic dispersion (photon-weighted RMS of $n$ across the band):
$$\sigma_n(b) = \sqrt{\int \tilde{w}_b(\lambda)\,[n(\lambda) - \langle n \rangle_b]^2\,d\lambda}$$

### Usage
Expected dipole length:
$$l_{\rm dip}(b,\,z)\;[{\rm arcsec}] = \sigma_n(b)\times\tan z\times\frac{180\times 3600}{\pi}$$

In [ ]:
def compute_sigma_n(band, lam=LAM, n_lam=N_LAM):
    """
    Chromatic dispersion of the refractive index in an LSST band.

    The integration weight is the **photon-count weight** for a flat f_nu SED:

        w(lambda) = T_b(lambda) / lambda

    which comes from:
        dN_gamma/dlambda  ∝  T_b * f_lambda * lambda/(hc)
                          =  T_b * (f_nu/lambda^2) * lambda/(hc)
                          ∝  T_b / lambda          (for f_nu = const)

    The dispersion is the normalised weighted RMS:

        sigma_n(b) = sqrt( <(n - <n>_b)^2>_b )

    where <.>_b denotes the w(lambda)-weighted average over the band.

    Parameters
    ----------
    band  : str
        One of 'u', 'g', 'r', 'i', 'z', 'y'.
    lam   : array_like
        Wavelength grid in **microns** (default: global LAM grid, 0.3–1.1 µm).
    n_lam : array_like
        Refractive index evaluated on *lam* (default: N_LAM from Ciddor).
        Can be n(λ) or (n(λ)-1); only *variations* within the band matter.

    Returns
    -------
    sigma_n : float
        Photon-weighted RMS chromatic spread of n(λ) across the band
        (dimensionless).

    Notes
    -----
    To obtain the expected DCR dipole length in arcsec:
        l_dip [arcsec] = compute_sigma_n(band) * tan(z) * RAD_TO_ARCSEC
    """
    lam = np.asarray(lam)
    n_lam = np.asarray(n_lam)

    # --- Filter throughput interpolated onto the common wavelength grid -------
    bp = LSST_FILTERS[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4  # Å → µm
    T = np.interp(lam, lam_bp, bp.response, left=0.0, right=0.0)

    # --- Photon-count weight: w(λ) = T(λ) / λ  (flat f_nu SED) --------------
    w_raw = T / lam

    # --- Normalise so that integral(w, lam) = 1 ------------------------------
    norm = simpson(w_raw, lam)
    if norm == 0.0:
        return 0.0
    w = w_raw / norm

    # --- Band-averaged refractive index --------------------------------------
    n_mean = simpson(w * n_lam, lam)

    # --- Photon-weighted variance  →  sigma_n --------------------------------
    var_n = simpson(w * (n_lam - n_mean) ** 2, lam)
    return np.sqrt(var_n)


# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"{'Band':>5}   {'sigma_n':>12}   {'l_dip(z=45°) [arcsec]':>22}")
print("-" * 46)
for b in BANDS:
    sn = compute_sigma_n(b)
    ldip_45 = sn * np.tan(np.deg2rad(45.0)) * RAD_TO_ARCSEC
    print(f"  {b}      {sn:.4e}          {ldip_45:.5f}")

## Observing geometry

### Zenith angle as a function of hour angle

From the standard spherical-astronomy relations with the observatory latitude $\phi$,
the declination $\delta$ of the field, and the hour angle $H$:

$$\cos z = \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H$$

hence:

$$\tan z = \frac{\sqrt{1 - \cos^2 z}}{\cos z}$$

### Parallactic angle as a function of hour angle

$$\tan q = \frac{\sin H}{\tan\phi\,\cos\delta - \sin\delta\,\cos H}$$

The parallactic angle $q$ gives the **orientation** of the dipole on sky (along the
direction of atmospheric dispersion, i.e. toward the zenith).

In [ ]:
def tanz_vs_HA(HA_hours, coords, location):
    """
    Compute tan(z) as a function of hour angle for a given field and observatory.

    Parameters
    ----------
    HA_hours : array_like
        Hour angle in **hours** (scalar or 1-D array).
    coords   : SkyCoord
        Sky coordinates of the target field.
    location : EarthLocation
        Observatory location.

    Returns
    -------
    tanz : ndarray
        tan(zenith angle) for each HA value.
    """
    HA_deg = np.asarray(HA_hours) * 15.0  # hours → degrees
    HA = np.deg2rad(HA_deg)
    dec = np.deg2rad(coords.dec.to(u.deg).value)
    lat = np.deg2rad(location.lat.to(u.deg).value)

    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    sinz = np.sqrt(np.clip(1.0 - cosz**2, 0.0, None))
    # Avoid division by zero at transit (cosz ~ 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        tanz = np.where(np.abs(cosz) > 1e-6, sinz / cosz, 0.0)
    return tanz


def parallactic_angle_vs_HA(HA_hours, coords, location):
    """
    Compute the parallactic angle q as a function of hour angle.

    Parameters
    ----------
    HA_hours : array_like
        Hour angle in **hours**.
    coords   : SkyCoord
        Sky coordinates of the target field.
    location : EarthLocation
        Observatory location.

    Returns
    -------
    q_deg : ndarray
        Parallactic angle in degrees.
    """
    HA_deg = np.asarray(HA_hours) * 15.0
    HA = np.deg2rad(HA_deg)
    dec = np.deg2rad(coords.dec.to(u.deg).value)
    lat = np.deg2rad(location.lat.to(u.deg).value)

    num = np.sin(HA)
    den = np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(HA)
    return np.degrees(np.arctan2(num, den))

## Pre-compute sigma_n for all bands

`compute_sigma_n` is purely spectro-photometric (no geometry), so we call it once
and cache the results in `SIGMA_N`.

In [ ]:
# sigma_n(b) for each LSST band — reusable in other notebooks
SIGMA_N = {b: compute_sigma_n(b) for b in BANDS}

print("sigma_n per band (photon-count weight, flat f_nu SED):")
for b, sn in SIGMA_N.items():
    print(f"  {b}:  {sn:.5e}")

## Intra-band chromatic dispersion of atmospheric refraction

### Motivation

The differential chromatic refraction (DCR) that produces dipole artifacts depends on the
**intra-band spread** of $n(\lambda)$.  To visualise this, the figure below shows:

- **Top panel** — $(n-1)\,(\lambda)$ computed from the Ciddor formula over the full LSST range,
  with one **error-bar point per band** at the photon-weighted pivot wavelength $\lambda_0(b)$,
  with vertical error $\pm\sigma_n(b)$ (the intra-band chromatic dispersion used for dipole
  length estimates).  Each point is drawn in the band's colour.

- **Bottom panel** — LSST $2023$ total throughput curves $T_b(\lambda)$, same colour code.

The two panels share the same $x$-axis so that the error bars align visually with the
transmission windows.  The relative size of the error bars illustrates why the **u** and
**g** bands produce larger DCR dipoles than the redder bands.


In [ ]:
# ── Figure: (n-1) vs lambda with per-band errorbar + LSST throughput ─────────
#
# Layout: GridSpec with 2 rows (height ratio 3:1.2), shared x-axis.
# Top panel  : (n-1) continuous curve + errorbar at photon-weighted pivot λ₀(b)
# Bottom panel: LSST 2023 throughput curves (filled), dashed lines at λ₀(b)

import matplotlib.gridspec as gridspec

# ── 1. Pre-compute per-band quantities ────────────────────────────────────────
# For each band: photon-weighted pivot wavelength λ₀, band-averaged (n-1), σ_n
band_stats = {}  # {band: (lambda0_nm, n_mean-1, sigma_n)}

for band in BAND_ORDER:
    lam_um = LAM.copy()  # µm grid
    n_lam = N_LAM.copy()  # n(λ)

    bp = LSST_FILTERS[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4  # Å → µm
    T = np.interp(lam_um, lam_bp, bp.response, left=0.0, right=0.0)

    # photon-count weight  w = T/λ  (flat f_ν SED)
    w_raw = T / lam_um
    norm = simpson(w_raw, lam_um)
    if norm == 0.0:
        continue
    w = w_raw / norm

    # pivot wavelength: photon-weighted mean λ
    lam0_um = simpson(w * lam_um, lam_um)  # µm
    lam0_nm = lam0_um * 1e3  # nm  (for x-axis)

    # band-averaged (n-1)
    n_mean = simpson(w * n_lam, lam_um)
    n_minus1 = n_mean - 1.0

    # σ_n already cached
    sn = SIGMA_N[band]

    band_stats[band] = (lam0_nm, n_minus1, sn)


# ── 2. Build figure with GridSpec ─────────────────────────────────────────────
fig = plt.figure(figsize=(9, 6.5))
gs = gridspec.GridSpec(
    2,
    1,
    height_ratios=[3, 1.3],
    hspace=0.06,
)

ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

# Convert LAM to nm for both panels
lam_nm = LAM * 1e3  # µm → nm
n_minus1_lam = (N_LAM - 1.0) * 1e6  # (n-1) × 10⁶ for readability

# ── Top panel: continuous (n-1) curve ─────────────────────────────────────────
ax_top.plot(
    lam_nm,
    n_minus1_lam,
    color="0.30",
    lw=1.5,
    label=r"$(n-1)\times10^6$  Ciddor",
    zorder=2,
)

# ── Per-band errorbar at pivot wavelength ─────────────────────────────────────
for band in BAND_ORDER:
    lam0_nm, n_minus1_b, sn = band_stats[band]
    ax_top.errorbar(
        lam0_nm,
        n_minus1_b * 1e6,  # scale to ×10⁶
        yerr=sn * 1e6,  # σ_n ×10⁶
        fmt="o",
        color=BAND_COLORS[band],
        markersize=8,
        capsize=6,
        capthick=2.0,
        lw=2.0,
        label=f"{band}:  $\\lambda_0$={lam0_nm:.0f} nm,  $\\sigma_n$={sn:.2e}",
        zorder=5,
    )

ax_top.set_ylabel(r"$(n-1)\times10^6$", fontsize=10)
ax_top.set_title(
    r"Atmospheric refraction dispersion $(n-1)$ and intra-band spread $\sigma_n$ across LSST bands",
    fontsize=12,
)
ax_top.legend(fontsize=7.5, ncol=3, loc="upper right", framealpha=0.88)
ax_top.tick_params(labelbottom=False)  # hide x tick labels on top panel

# ── Bottom panel: LSST throughput curves (filled) ─────────────────────────────
for band in BAND_ORDER:
    bp = LSST_FILTERS[f"lsst2023-{band}"]
    x_nm = bp.wavelength * 0.1  # Å → nm
    y_resp = bp.response
    ax_bot.fill_between(x_nm, y_resp, alpha=0.30, color=BAND_COLORS[band])
    ax_bot.plot(x_nm, y_resp, color=BAND_COLORS[band], lw=1.3, label=band)
    # dashed vertical line at photon-weighted pivot λ₀(b)
    ax_bot.axvline(
        band_stats[band][0],
        color=BAND_COLORS[band],
        lw=0.9,
        ls="--",
        alpha=0.8,
    )

ax_bot.set_xlabel(r"Wavelength $\lambda$ (nm)", fontsize=12)
ax_bot.set_ylabel("Throughput", fontsize=12)
ax_bot.legend(
    title="LSST band",
    fontsize=8,
    ncol=1,
    loc="upper right",
    framealpha=0.88,
    title_fontsize=8,
    bbox_to_anchor=(1.05, 1.05),
)
ax_bot.set_ylim(0, None)

# Shared x range
ax_top.set_xlim(300, 1100)

plt.tight_layout()
savefig("fig_n_minus1_intraband_dispersion")
plt.show()

## Expected dipole length vs hour angle — 2×3 subplot grid per DDF set

For each field we show $l_{\rm dip}(b, H) = \sigma_n(b) \times \tan z(H)$ in arcseconds
for $H \in [-6\,{\rm h},\, +6\,{\rm h}]$, with one curve per LSST band.

We split the 7 DDFs into two figures:
- **Figure 1**: COSMOS, ELAIS-S1, ECDFS, EDFS-a, EDFS-b, EDFS  (2×3)
- **Figure 2**: M49 alone (remaining panels hidden)

In [ ]:
# Hour angle grid:  -6 h → +6 h
HA = np.linspace(-6.0, 6.0, 500)  # hours

In [ ]:
def plot_dipole_length_grid(field_names, title_tag, fig_name):
    """
    Plot the expected DCR dipole length l_dip = sigma_n(b) * tan(z) [arcsec]
    vs hour angle for a list of DDF fields, arranged in a 2x3 subplot grid.

    Parameters
    ----------
    field_names : list of str
        DDF names (up to 6) from DEEP_FIELDS.
    title_tag   : str
        Suptitle suffix.
    fig_name    : str
        Base filename for savefig (without extension).
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 7), sharex=True)
    fig.suptitle(
        f"DCR-induced dipole length  $l_{{\\rm dip}} = \\sigma_n(b)\\,\\tan z$  [{title_tag}]",
        fontsize=11,
    )

    axes_flat = axes.flatten()

    for idx, field_name in enumerate(field_names):
        ax = axes_flat[idx]
        ra_deg, dec_deg = DEEP_FIELDS[field_name]
        coords = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

        tanz = tanz_vs_HA(HA, coords, RUBIN_LOCATION)

        for b in BANDS:
            l_dip_arcsec = SIGMA_N[b] * tanz * RAD_TO_ARCSEC
            ax.plot(HA, l_dip_arcsec, color=BAND_COLORS[b], lw=1.4, label=b if idx == 0 else None)

        ax.set_title(f"{field_name}\nRA={ra_deg:.1f}°  Dec={dec_deg:.1f}°", fontsize=8)
        ax.set_xlabel("Hour angle H [h]")
        ax.set_ylabel(r"$l_{\rm dip}$ [arcsec]")
        ax.set_xlim(-6, 6)
        ax.set_ylim(0.0, 1.2)
        ax.axvline(0, color="k", lw=0.6, ls="--", alpha=0.4)  # transit

    # Hide unused subplots
    for idx in range(len(field_names), nrows * ncols):
        axes_flat[idx].set_visible(False)

    # Single legend on the first panel
    axes_flat[0].legend(title="band", fontsize=7, title_fontsize=7, loc="upper right", ncol=2)

    plt.tight_layout()
    savefig(fig_name)
    plt.show()

In [ ]:
# Figure 1:  first 6 DDFs
# DDF_SET1 = ["COSMOS", "ECDFS", "EDFS-a", "EDFS-b", "EDFS", "M49"]
DDF_SET1 = ["COSMOS", "ECDFS", "EDFS"]
plot_dipole_length_grid(DDF_SET1, title_tag="DDFs set 1", fig_name="fig_dipole_length_DDFs_set1")

In [ ]:
# Figure 2:  M49 (shown alone; remaining panels hidden)
DDF_SET2 = ["COSMOS"]
plot_dipole_length_grid(DDF_SET2, title_tag="DDFs set 2", fig_name="fig_dipole_length_DDFs_set2")

## Summary table: sigma_n and reference dipole lengths

$l_{\rm dip}$ evaluated at representative zenith angles (30°, 45°, 60°).

In [ ]:
z_ref = [30.0, 45.0, 60.0]  # reference zenith angles in degrees

rows = []
for b in BANDS:
    sn = SIGMA_N[b]
    row = {"band": b, "sigma_n": sn}
    for z in z_ref:
        ldip = sn * np.tan(np.deg2rad(z)) * RAD_TO_ARCSEC
        row[f"l_dip(z={z:.0f}°) [arcsec]"] = ldip
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index("band")
df_summary.style.format({"sigma_n": "{:.4e}", **{c: "{:.5f}" for c in df_summary.columns if "l_dip" in c}})

---
### How to reuse `compute_sigma_n` in another notebook

```python
# Prerequisites: LAM, N_LAM, LSST_FILTERS must be defined (see cells above).
# Weight: w(lambda) = T_b(lambda) / lambda   [flat f_nu SED, photon counts]

sigma_n_u = compute_sigma_n('u')        # dimensionless float
l_dip_arcsec = sigma_n_u * np.tan(np.deg2rad(z_deg)) * RAD_TO_ARCSEC
```

All six values are also cached in the `SIGMA_N` dict:
```python
SIGMA_N = {b: compute_sigma_n(b) for b in 'ugrizy'}
```